# Embedding 提出用ノートブック（次元削減 8 パターン）

**基準（Public 最高）**: **movie_title_info（small）+ PCA 16** → **Public 0.75449**（embedding 種別×PCA 8/16 の提出比較で最良）。  
この設定の **embedding を固定** し、**次元削減手法だけ** を 8 パターンに変えて提出ファイルを作成する。結果の記録は `docs/05_EMBEDDING_SUBMISSION_RESULTS.md`。

| # | 手法 | 説明 |
|---|------|------|
| 1 | **pca** | PCA（分散最大化）← 基準 0.75449 |
| 2 | **truncated_svd** | TruncatedSVD（SVD、中心化なし） |
| 3 | **ica** | ICA（独立成分） |
| 4 | **random_projection** | Gaussian Random Projection |
| 5 | **sparse_pca** | Sparse PCA |
| 6 | **nmf** | NMF（非負値化して適用） |
| 7 | **kernel_pca** | Kernel PCA（RBF） |
| 8 | **umap** | UMAP（requirements.txt に umap-learn あり） |

**手順**: 設定セルで `REDUCTION_METHODS` と `N_COMPONENTS` を編集 → 上から実行 → `outputs/submissions/` に各手法ごとの CSV が保存される。

**追加実験（§4）**: 残り提出回数用に、シードアベレージ・PCA24・2本アンサンブル・TruncatedSVD を `EXTRA_EXPERIMENTS` で選んで実行できる。

## 1. 設定

In [1]:
# ========== 提出用設定（ここを編集） ==========
# 最良 embedding を固定（Public 0.75449: movie_title_info + PCA 16）
EMBEDDING_NAME = "movie_title_info"

# 次元削減後の次元数
N_COMPONENTS = 16

# 試す次元削減手法（このリストの分だけ提出ファイルができる）
# 8 パターン全部: pca, truncated_svd, ica, random_projection, sparse_pca, nmf, kernel_pca, umap
REDUCTION_METHODS = [
    "pca",
    "truncated_svd",
    "ica",
    "random_projection",
    "sparse_pca",
    "nmf",
    "kernel_pca",
    "umap",
]
# 例: PCA と SVD だけ試す → REDUCTION_METHODS = ["pca", "truncated_svd"]
# UMAP だけ使わない → REDUCTION_METHODS から "umap" を削除

# ---------- 追加実験（残り提出回数用、セクション 4 で実行） ----------
# 含めたいものだけリストに書く。[] なら追加実験なし。
EXTRA_EXPERIMENTS = [
    "seed_avg",        # シード 3 つで予測平均（movie_title_info + PCA 16）
    "pca24",           # movie_title_info + PCA 24 次元
    "ensemble_two",    # 1位・2位の提出 2 本を 0.5:0.5 で平均
    "truncated_svd",   # movie_title_info + TruncatedSVD 16（次元削減の別手法 1 本）
]
# ==============================================================

In [2]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from lib import get_baseline_data, BASELINE_FEATURES, BASELINE_LGB_PARAMS
from lib.embedding_reduction import (
    REDUCTION_METHODS as ALL_REDUCTION_METHODS,
    fit_transform_embedding,
)
from lib.submission import save_submission, verify_submission

OUTPUT_DIR = Path("outputs")
EMBEDDINGS_DIR = OUTPUT_DIR / "embeddings"
SUBMISSIONS_DIR = OUTPUT_DIR / "submissions"
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

EMBEDDING_CONFIGS = {
    "movie_info": {"path": EMBEDDINGS_DIR / "movie_info_embeddings.pkl", "loader": "movie_info"},
    "movie_info_large": {"path": EMBEDDINGS_DIR / "movie_info_embeddings_large.pkl", "loader": "movie_info"},
    "movie_title_info": {"path": EMBEDDINGS_DIR / "movie_title_info_embeddings.pkl", "loader": "title_info"},
    "movie_title_info_large": {"path": EMBEDDINGS_DIR / "movie_title_info_embeddings_large.pkl", "loader": "title_info"},
}

def load_embeddings(name: str):
    if name not in EMBEDDING_CONFIGS:
        raise ValueError(f"不明な embedding: {name}. 利用可能: {list(EMBEDDING_CONFIGS.keys())}")
    config = EMBEDDING_CONFIGS[name]
    path = Path(config["path"])
    if not path.exists():
        alt = path.with_suffix(".parquet")
        if alt.exists():
            path = alt
        else:
            raise FileNotFoundError(f"embedding ファイルがありません: {path} または {alt}")
    if config["loader"] == "movie_info":
        from lib.openai_embeddings import load_movie_info_embeddings
        return load_movie_info_embeddings(path=path)
    from lib.openai_embeddings import load_movie_title_info_embeddings
    return load_movie_title_info_embeddings(path=path)

def _merge_embeddings(df, emb_df):
    emb_cols = [c for c in emb_df.columns if c != "rotten_tomatoes_link"]
    merged = df[["rotten_tomatoes_link"]].merge(
        emb_df[["rotten_tomatoes_link"] + emb_cols], on="rotten_tomatoes_link", how="left"
    )
    return merged[emb_cols].fillna(0).astype(np.float32).values

for m in REDUCTION_METHODS:
    if m not in ALL_REDUCTION_METHODS:
        raise ValueError(f"不明な次元削減: {m}. 利用可能: {ALL_REDUCTION_METHODS}")
print(f"embedding: {EMBEDDING_NAME}, 次元: {N_COMPONENTS}, 手法: {REDUCTION_METHODS}")
print(f"提出先: {SUBMISSIONS_DIR}")

embedding: movie_title_info, 次元: 16, 手法: ['pca', 'truncated_svd', 'ica', 'random_projection', 'sparse_pca', 'nmf', 'kernel_pca', 'umap']
提出先: outputs/submissions


## 2. データ読み込み・embedding 行列の準備

In [3]:
train, test = get_baseline_data()
y = train["target"].values
base_features = list(BASELINE_FEATURES)

emb_df = load_embeddings(EMBEDDING_NAME)
E_train = _merge_embeddings(train, emb_df)
E_test = _merge_embeddings(test, emb_df)

print(f"Train: {len(train):,}, Test: {len(test):,}, Embedding 次元: {E_train.shape[1]}")

Train: 653,507, Test: 40,716, Embedding 次元: 1536


## 3. 各次元削減で学習 → 予測 → 提出 CSV 保存

In [ ]:
safe_emb = EMBEDDING_NAME.replace(" ", "_")
saved = []

for method in REDUCTION_METHODS:
    fname_check = f"submission_embedding_{safe_emb}_{method}{N_COMPONENTS}.csv"
    if (SUBMISSIONS_DIR / fname_check).exists():
        print(f"  [{method}] スキップ（既存）")
        saved.append(fname_check)
        continue
    try:
        train_reduced, test_reduced, prefix = fit_transform_embedding(
            E_train, E_test, method=method, n_components=N_COMPONENTS, random_state=42
        )
    except Exception as e:
        print(f"  [{method}] スキップ: {e}")
        continue
    n_comp = train_reduced.shape[1]
    red_names = [f"{prefix}_{i}" for i in range(n_comp)]
    X_train = pd.concat([
        train[base_features].reset_index(drop=True),
        pd.DataFrame(train_reduced, columns=red_names),
    ], axis=1)
    X_test = pd.concat([
        test[base_features].reset_index(drop=True),
        pd.DataFrame(test_reduced, columns=red_names),
    ], axis=1)
    feats = base_features + red_names
    model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
    model.fit(X_train[feats], y)
    pred = model.predict_proba(X_test[feats])[:, 1]
    fname = f"submission_embedding_{safe_emb}_{method}{n_comp}.csv"
    out_path = SUBMISSIONS_DIR / fname
    save_submission(test, pred, out_path, sanitize=True)
    v = verify_submission(out_path, test)
    saved.append(fname)
    status = "OK" if v["ok"] else v["message"]
    print(f"  [{method}] → {fname}  ({status})")

print(f"\n保存先: {SUBMISSIONS_DIR}")
print(f"作成した提出ファイル: {len(saved)} 件")

  [pca] スキップ（既存）: submission_embedding_movie_title_info_pca16.csv
  [truncated_svd] スキップ（既存）: submission_embedding_movie_title_info_truncated_svd16.csv


## 4. 追加実験（残り提出回数用）

上で `EXTRA_EXPERIMENTS` に含めたものだけ実行する。**セクション 2（データ読み込み）を先に実行しておくこと。**

| キー | 内容 | 提出 1 本 |
|------|------|-----------|
| `seed_avg` | movie_title_info + PCA 16 をシード 42,43,44 で 3 本学習 → 予測を平均 | ○ |
| `pca24` | movie_title_info + PCA **24** 次元で 1 本 | ○ |
| `ensemble_two` | 既存の movie_title_info PCA16 と movie_title_info_large PCA16 の CSV を 0.5:0.5 で平均 | ○ |
| `truncated_svd` | movie_title_info + TruncatedSVD 16 で 1 本（次元削減の別手法） | ○ |

In [ ]:
if not EXTRA_EXPERIMENTS:
    print("EXTRA_EXPERIMENTS が空のためスキップしました。")
else:
    extra_saved = []
    safe_emb = EMBEDDING_NAME.replace(" ", "_")

    if "seed_avg" in EXTRA_EXPERIMENTS:
        seeds = [42, 43, 44]
        preds = []
        for rs in seeds:
            train_r, test_r, prefix = fit_transform_embedding(
                E_train, E_test, method="pca", n_components=N_COMPONENTS, random_state=rs
            )
            n_comp = train_r.shape[1]
            red_names = [f"{prefix}_{i}" for i in range(n_comp)]
            X_tr = pd.concat([train[base_features].reset_index(drop=True), pd.DataFrame(train_r, columns=red_names)], axis=1)
            X_te = pd.concat([test[base_features].reset_index(drop=True), pd.DataFrame(test_r, columns=red_names)], axis=1)
            model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
            model.fit(X_tr[base_features + red_names], y)
            preds.append(model.predict_proba(X_te[base_features + red_names])[:, 1])
        pred_avg = np.mean(preds, axis=0)
        fname = f"submission_embedding_{safe_emb}_pca{N_COMPONENTS}_seed_avg.csv"
        save_submission(test, pred_avg, SUBMISSIONS_DIR / fname, sanitize=True)
        v = verify_submission(SUBMISSIONS_DIR / fname, test)
        extra_saved.append(fname)
        print(f"  [seed_avg] → {fname}  ({'OK' if v['ok'] else v['message']})")

    if "pca24" in EXTRA_EXPERIMENTS:
        train_r, test_r, prefix = fit_transform_embedding(
            E_train, E_test, method="pca", n_components=24, random_state=42
        )
        red_names = [f"{prefix}_{i}" for i in range(train_r.shape[1])]
        X_tr = pd.concat([train[base_features].reset_index(drop=True), pd.DataFrame(train_r, columns=red_names)], axis=1)
        X_te = pd.concat([test[base_features].reset_index(drop=True), pd.DataFrame(test_r, columns=red_names)], axis=1)
        model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
        model.fit(X_tr[base_features + red_names], y)
        pred = model.predict_proba(X_te[base_features + red_names])[:, 1]
        fname = f"submission_embedding_{safe_emb}_pca24.csv"
        save_submission(test, pred, SUBMISSIONS_DIR / fname, sanitize=True)
        v = verify_submission(SUBMISSIONS_DIR / fname, test)
        extra_saved.append(fname)
        print(f"  [pca24] → {fname}  ({'OK' if v['ok'] else v['message']})")

    if "ensemble_two" in EXTRA_EXPERIMENTS:
        p1 = SUBMISSIONS_DIR / "submission_embedding_movie_title_info_pca16.csv"
        p2 = SUBMISSIONS_DIR / "submission_embedding_movie_title_info_large_pca16.csv"
        if p1.exists() and p2.exists():
            d1 = pd.read_csv(p1)
            d2 = pd.read_csv(p2)
            merged = d1[["ID", "target"]].merge(d2[["ID", "target"]], on="ID", suffixes=("_1", "_2"))
            merged["target"] = (merged["target_1"].astype(float) + merged["target_2"].astype(float)) / 2
            test_with_pred = test[["ID"]].merge(merged[["ID", "target"]], on="ID", how="left")
            if test_with_pred["target"].isna().any():
                print("  [ensemble_two] スキップ: test の ID が 2 本の CSV と一致しません")
            else:
                pred_avg = test_with_pred["target"].values
                fname = "submission_embedding_ensemble_movie_title_info_and_large_pca16.csv"
                save_submission(test, pred_avg, SUBMISSIONS_DIR / fname, sanitize=True)
                v = verify_submission(SUBMISSIONS_DIR / fname, test)
                extra_saved.append(fname)
                print(f"  [ensemble_two] → {fname}  ({'OK' if v['ok'] else v['message']})")
        else:
            print(f"  [ensemble_two] スキップ: 2 本の CSV がありません（{p1.name}, {p2.name} を先に作成）")

    if "truncated_svd" in EXTRA_EXPERIMENTS:
        train_r, test_r, prefix = fit_transform_embedding(
            E_train, E_test, method="truncated_svd", n_components=N_COMPONENTS, random_state=42
        )
        red_names = [f"{prefix}_{i}" for i in range(train_r.shape[1])]
        X_tr = pd.concat([train[base_features].reset_index(drop=True), pd.DataFrame(train_r, columns=red_names)], axis=1)
        X_te = pd.concat([test[base_features].reset_index(drop=True), pd.DataFrame(test_r, columns=red_names)], axis=1)
        model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
        model.fit(X_tr[base_features + red_names], y)
        pred = model.predict_proba(X_te[base_features + red_names])[:, 1]
        fname = f"submission_embedding_{safe_emb}_truncated_svd{N_COMPONENTS}.csv"
        save_submission(test, pred, SUBMISSIONS_DIR / fname, sanitize=True)
        v = verify_submission(SUBMISSIONS_DIR / fname, test)
        extra_saved.append(fname)
        print(f"  [truncated_svd] → {fname}  ({'OK' if v['ok'] else v['message']})")

    print(f"\n追加実験で作成: {len(extra_saved)} 件")